In [1]:
import json

# Read the JSON file
with open("data/gods.json", "r", encoding="utf-8") as f:
    gods = json.load(f)

# Print the count and the first element
print("Length of gods:", len(gods))
print("First item:", gods[0])


Length of gods: 14
First item: {'name': 'زئوس', 'description': 'زئوس بر کوه الیمپوس حکومت می\u200cکند و پادشاه خدایان است. او پسر کرونوس و ریا، شوهر هرا است، و 15 کودک دارد که همه آنها المپیایی هستند.', 'appearance': 'او به عنوان یک مرد قد بلند و ستبر با لباس فاخر نشان داده شده است. او دارای موی بلند و ریش بلند است. به طور کلی، او به صورت ایستاده به تصویر درآمده است.', 'features': 'رعد، اسلحه او و ویژگی اصلی وی است که به طور کلی در دست راست او قرار دارد. گاهی اوقات او یک عصای سلطنتی را در دست چپ خود حمل می\u200cکند. گاو و عقاب حیوانات مقدس او هستند.'}


In [2]:
import json

# Convert the gods object to a JSON string (ensure_ascii=False preserves Persian characters)
gods_context = json.dumps(gods, ensure_ascii=False, indent=2)

system_message = f"""شما یک دستیار هوشمند در زمینه اساطیر یونان هستید.

اطلاعات مرجع:
{gods_context}

دستورالعمل‌های الزامی:
۱. به تمام سوالات کاربران درباره خدایان یونان، صرفاً بر اساس اطلاعات مرجع بالا و به زبان فارسی پاسخ دهید.
۲. پاسخ‌ها باید بسیار کوتاه باشند (حداکثر ۴ الی ۵ کلمه).
۳. اگر پاسخ سوال را در متن نیافتید یا نمی‌توانید در حداکثر ۵ کلمه پاسخ دهید، دقیقاً و صرفاً کلمه «نمیدانم» را خروجی دهید.
"""


In [3]:
examples = [
    {"question": "پادشاه خدایان یونان کیست؟", "answer": "زئوس"},
    {"question": "آتنا حامی کدام شهر است؟", "answer": "شهر آتن"},
    {"question": "مادر زئوس چه نام دارد؟", "answer": "ریا"},
    {"question": "پوسوئیدون به چه نام دیگری معروف است؟", "answer": "نپتون"},
    {"question": "آگاه‌ترین خدای یونانی چه کسی است؟", "answer": "آپولون"},
    {"question": "کدام خدا آرام‌ترین خدایان محسوب می‌شود؟", "answer": "دمتر"},
    {"question": "زئوس در دست چپ خود گاهی چه چیزی حمل می‌کند؟", "answer": "عصای سلطنتی"},
    {"question": "پدر و مادر آرس چه کسانی هستند؟", "answer": "زئوس و هرا"},
    {"question": "آتنا از کدام عضو بدن زئوس متولد شد؟", "answer": "سر زئوس"},
    {"question": "هفایستوس بر روی چه حیوانی می‌نشیند؟", "answer": "الاغ"},
]


In [4]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Load environment variables (automatically finds .env in the parent directory)
load_dotenv(find_dotenv())

# 2. Initialize Gemini with temperature=0 for deterministic outputs
model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",  # or "gemini-flash-lite-latest"
    temperature=0
)

# 3. Define the template for individual few-shot examples
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}"),
    ("ai", "{answer}"),
])

# 4. Construct the few-shot prompt
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# 5. Assemble the final prompt
# Using SystemMessage(content=system_message) avoids KeyError with JSON curly braces
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=system_message),
    few_shot_prompt,
    ("human", "{question}"),
])


In [ ]:
import time
import pandas as pd
from langchain_core.output_parsers import StrOutputParser

# 1. Create the chain (using StrOutputParser to cleanly extract the string output)
chain = prompt | model | StrOutputParser()

# 2. Read questions from data/questions.csv
questions = pd.read_csv("data/questions.csv")

# 3 & 4. Iterate over questions, invoke chain, and handle rate limits
answers = []
total_questions = len(questions)

for i, q in enumerate(questions["question"]):
    # Invoke chain with the question
    response = chain.invoke({"question": q})
    
    # Clean output and store
    answer_text = response.strip() if isinstance(response, str) else response.content.strip()
    answers.append(answer_text)
    
    print(f"[{i + 1}/{total_questions}] {q} -> {answer_text}")
    
    # Respect API rate limits: sleep 65s after every 10 requests (except the last batch)
    if (i + 1) % 10 == 0 and (i + 1) < total_questions:
        print("⏳ Rate limit pause: Sleeping for 65 seconds...")
        time.sleep(65)

# 5. Store all answers in exact original order in the submission DataFrame
submission = pd.DataFrame({"answer": answers})

# Preview the results
print("\nSubmission DataFrame preview:")
print(submission.head(10))


ReadError: [WinError 10053] An established connection was aborted by the software in your host machine

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '\n<html><head>\n<meta http-equiv="content-type" content="text/html;charset=utf-8">\n<title>403 Forbidden</title>\n</head>\n<body text=#000000 bgcolor=#ffffff>\n<h1>Error: Forbidden</h1>\n<h2>Your client does not have permission to get URL <code>/runs/multipart</code> from this server.</h2>\n<h2></h2>\n</body></html>\n')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '\n<html><head>\n<meta http-equiv="content-type" content="text/html;charset=utf-8">\n<title>403 Forbidden</title>\n</head>\n<body text=#000000 bgcolor=#ffffff>\n<h1>Error: Forbidden</h1>\n<h2>Y